# Phase 4 — LLM Extraction & Three-Model Comparison

In [1]:
# ── LLM Configuration ──────────────────────────────────────────────────────
# Model is set via GEMINI_MODEL in ../.env
# Supported: gemini-3.1-flash-image-preview | gemini-3-pro-image-preview
# ───────────────────────────────────────────────────────────────────────────

In [2]:
import numpy as np
import pandas as pd
import os
import json
import time
import gc
import pickle
import warnings
warnings.filterwarnings('ignore')

from sklearn.preprocessing import LabelEncoder
from sklearn.impute import SimpleImputer
from sklearn.model_selection import KFold
from sklearn.metrics import roc_auc_score, confusion_matrix
from sklearn.calibration import calibration_curve
import lightgbm as lgb
import matplotlib.pyplot as plt
import seaborn as sns

from dotenv import load_dotenv
load_dotenv('../.env')

True

In [3]:
from google import genai

ALLOWED_MODELS = {"gemini-3.1-flash-image-preview", "gemini-3-pro-image-preview"}

MODEL_NAME = os.environ.get("GEMINI_MODEL", "gemini-3.1-flash-image-preview")
assert MODEL_NAME in ALLOWED_MODELS, f"GEMINI_MODEL must be one of {ALLOWED_MODELS}, got '{MODEL_NAME}'"

LLM_CLIENT = genai.Client(api_key=os.environ.get("GOOGLE_API_KEY"))

def retry_with_backoff(func, max_retries=5, initial_delay=1.0, multiplier=2.0):
    delay = initial_delay
    for attempt in range(max_retries):
        try:
            return func()
        except Exception as e:
            if attempt == max_retries - 1: raise
            print(f"  Attempt {attempt+1} failed: {e}. Retrying in {delay:.1f}s...")
            time.sleep(delay)
            delay *= multiplier

def extract_text_via_llm(prompt):
    def _call():
        response = LLM_CLIENT.models.generate_content(model=MODEL_NAME, contents=prompt)
        return response.text
    return retry_with_backoff(_call)

print(f"Using Gemini model: {MODEL_NAME}")

Using Gemini model: gemini-3.1-flash-image-preview


## Step 4a — LLM Feature Extraction

In [4]:
from google.genai import types


def _strip_fences(raw):
    raw = raw.strip()
    if raw.startswith('```'): raw = raw.split('\n', 1)[1]
    if raw.endswith('```'): raw = raw.rsplit('```', 1)[0]
    return raw


def _extract_from_image(sk_id, path, prompt, doc_type):
    """Shared helper for image-based extraction (paystub, linkedin, id)."""
    if not os.path.exists(path): return None
    with open(path, 'rb') as f:
        img_data = f.read()
    contents = [types.Content(parts=[
        types.Part.from_bytes(data=img_data, mime_type="image/png"),
        types.Part.from_text(text=prompt)
    ])]
    def _call():
        response = LLM_CLIENT.models.generate_content(model=MODEL_NAME, contents=contents)
        return response.text
    raw = retry_with_backoff(_call)
    try:
        data = json.loads(_strip_fences(raw))
        data['SK_ID_CURR'] = sk_id
        data['doc_type'] = doc_type
        return data
    except:
        return {'SK_ID_CURR': sk_id, 'doc_type': doc_type, 'parse_error': True}


def _extract_from_pdf(sk_id, path, prompt_body, doc_type):
    """Shared helper for PDF-based extraction (bank_statement, property, credit_report).
    Sends the raw PDF bytes to Gemini so the model reads the rendered document."""
    if not os.path.exists(path): return None
    with open(path, 'rb') as f:
        pdf_data = f.read()
    contents = [types.Content(parts=[
        types.Part.from_bytes(data=pdf_data, mime_type="application/pdf"),
        types.Part.from_text(text=prompt_body)
    ])]
    def _call():
        response = LLM_CLIENT.models.generate_content(model=MODEL_NAME, contents=contents)
        return response.text
    raw = retry_with_backoff(_call)
    try:
        data = json.loads(_strip_fences(raw))
        data['SK_ID_CURR'] = sk_id
        data['doc_type'] = doc_type
        return data
    except:
        return {'SK_ID_CURR': sk_id, 'doc_type': doc_type, 'parse_error': True}


def extract_from_paystub(sk_id):
    return _extract_from_image(sk_id,
        f'../unstructured_data/paystubs/{sk_id}_paystub.png',
        """Analyze this paystub image and extract the following fields as JSON with a confidence score (1-5) per field:
- gross_pay (number)
- net_pay (number)
- net_to_gross_ratio (number, net/gross)
- tenure_years (number, from hire date)
- missing_field_indicator (boolean, true if retirement/401k deduction is absent)
- ocr_noise_score (number 0-1, estimated document quality, 1=clean)

Return ONLY valid JSON, no commentary.""", 'paystub')


def extract_from_linkedin(sk_id):
    return _extract_from_image(sk_id,
        f'../unstructured_data/linkedin/{sk_id}_linkedin.png',
        """Analyze this LinkedIn profile screenshot and extract:
- job_hop_count (number of different employers)
- seniority_score (1-6: 1=Entry, 2=Associate, 3=Mid, 4=Senior, 5=Manager/Director, 6=VP+)
- skill_count (number of skills listed)
- career_progression_index (0-1, promotions vs lateral moves)
- connections_estimate (number)

Return ONLY valid JSON with confidence (1-5) per field, no commentary.""", 'linkedin')


def extract_from_bank_statement(sk_id):
    return _extract_from_pdf(sk_id,
        f'../unstructured_data/bank_statements/{sk_id}_bank_statement.pdf',
        """Analyze this bank statement PDF and extract the following fields as JSON with a confidence score (1-5) per field:
- monthly_income (number)
- monthly_repayment (number)
- overdue_flag (boolean)
- credit_sum (total outstanding credit)
- credit_debt (outstanding debt)

Return ONLY valid JSON with confidence (1-5) per field.""", 'bank_statement')


def extract_from_id_document(sk_id):
    return _extract_from_image(sk_id,
        f'../unstructured_data/id_documents/{sk_id}_id.png',
        """Extract from this ID card image:
- gender (M/F)
- date_of_birth (YYYY-MM-DD)
- family_status (string)
- cnt_children (number)

Return ONLY valid JSON with confidence (1-5) per field.""", 'id_document')


def extract_from_property_doc(sk_id):
    return _extract_from_pdf(sk_id,
        f'../unstructured_data/property_docs/{sk_id}_property.pdf',
        """Analyze this property/utility document PDF and extract:
- housing_type (string)
- own_realty (Y/N)
- own_car (Y/N)
- car_age (number, years)
- region_rating (1-3)
- living_area (number, sq ft)
- total_area (number, sq ft)

Return ONLY valid JSON with confidence (1-5) per field.""", 'property_doc')


def extract_from_credit_report(sk_id):
    return _extract_from_pdf(sk_id,
        f'../unstructured_data/credit_reports/{sk_id}_credit_report.pdf',
        """Analyze this credit bureau report PDF and extract the following fields as JSON.

The report contains three credit scores in the 300-850 range. Convert each score
to a 0-1 value using the formula: ext_source = (score - 300) / 550

- ext_source_1: apply (first_score - 300) / 550 to get a value between 0 and 1
- ext_source_2: apply (second_score - 300) / 550 to get a value between 0 and 1
- ext_source_3: apply (third_score - 300) / 550 to get a value between 0 and 1
- bureau_inquiries_hour (number)
- bureau_inquiries_day (number)
- bureau_inquiries_week (number)
- bureau_inquiries_month (number)
- bureau_inquiries_quarter (number)
- bureau_inquiries_year (number)

Return ONLY valid JSON with confidence (1-5) per field.""", 'credit_report')

In [5]:
import re as _re

app_train_full = pd.read_csv('../datasets/application_train.csv')

# Discover all SK_IDs from unstructured data file names
sample_ids = sorted({
    int(_re.match(r'^(\d+)_', f).group(1))
    for root, _, files in os.walk('../unstructured_data')
    for f in files if _re.match(r'^(\d+)_', f)
})
print(f"Found {len(sample_ids)} customer IDs in unstructured_data/")

from dotenv import load_dotenv
load_dotenv('../.env')
EXTRACT_DATA = os.environ.get("EXTRACT_DATA", "false").lower() == "true"
print(f"EXTRACT_DATA = {EXTRACT_DATA}")

EXTRACTORS = {
    'paystub': extract_from_paystub,
    'linkedin': extract_from_linkedin,
    'bank_statement': extract_from_bank_statement,
    'id_document': extract_from_id_document,
    'property_doc': extract_from_property_doc,
    'credit_report': extract_from_credit_report,
}

log_path = '../unstructured_data/extraction_results.jsonl'

if not EXTRACT_DATA and os.path.exists(log_path):
    print(f"EXTRACT_DATA=false — loading existing extraction_results.jsonl")
    extraction_results = [json.loads(line) for line in open(log_path)]
    parse_success = sum(1 for r in extraction_results if not r.get('parse_error'))
    parse_fail   = sum(1 for r in extraction_results if r.get('parse_error'))
else:
    extraction_results = []
    parse_success = 0
    parse_fail = 0

    for sk_id in sample_ids:
        print(f"Extracting for SK_ID_CURR={sk_id}...")
        for doc_type, extractor in EXTRACTORS.items():
            result = extractor(sk_id)
            if result is not None:
                extraction_results.append(result)
                if result.get('parse_error'):
                    parse_fail += 1
                else:
                    parse_success += 1

    with open(log_path, 'w') as f:
        for r in extraction_results:
            f.write(json.dumps(r, default=str) + '\n')
    print(f"Saved extraction results to {log_path}")

total = parse_success + parse_fail
print(f"\nExtraction: {parse_success}/{total} valid JSON ({parse_success/total*100:.1f}%)")
print(f"Target: >= 90%")

Found 110 customer IDs in unstructured_data/
EXTRACT_DATA = False
EXTRACT_DATA=false — loading existing extraction_results.jsonl

Extraction: 656/660 valid JSON (99.4%)
Target: >= 90%


## Step 4b — Dataset Reconstruction

In [6]:
trimmed_full = pd.read_csv('../datasets/application_train_trimmed.csv')
print(f"Trimmed dataset (full): {trimmed_full.shape}")

trimmed_sample = trimmed_full[trimmed_full['SK_ID_CURR'].isin(sample_ids)].copy()
trimmed_train  = trimmed_full[~trimmed_full['SK_ID_CURR'].isin(sample_ids)].copy()
print(f"Held-out test set ({len(sample_ids)} customers): {trimmed_sample.shape}")
print(f"Training set (rest): {trimmed_train.shape}")

def _flatten_value(v):
    """Extract numeric/scalar value from dict-wrapped extraction results."""
    if isinstance(v, dict):
        if 'value' in v:
            return v['value']
        return np.nan
    if isinstance(v, list) and len(v) >= 1:
        return v[0]
    return v

flat_results = []
for rec in extraction_results:
    flat_rec = {}
    for k, val in rec.items():
        flat_rec[k] = _flatten_value(val)
    flat_results.append(flat_rec)

# ── Map extracted values back to original column names ──────────────────────
# This mapping connects (doc_type, extraction_field) to the original structured
# column that was trimmed, with an optional numeric scale factor.
EXTRACTION_TO_ORIGINAL = {
    ('credit_report',  'ext_source_1'):           ('EXT_SOURCE_1',                 1),
    ('credit_report',  'ext_source_2'):           ('EXT_SOURCE_2',                 1),
    ('credit_report',  'ext_source_3'):           ('EXT_SOURCE_3',                 1),
    ('credit_report',  'bureau_inquiries_hour'):  ('AMT_REQ_CREDIT_BUREAU_HOUR',   1),
    ('credit_report',  'bureau_inquiries_day'):   ('AMT_REQ_CREDIT_BUREAU_DAY',    1),
    ('credit_report',  'bureau_inquiries_week'):  ('AMT_REQ_CREDIT_BUREAU_WEEK',   1),
    ('credit_report',  'bureau_inquiries_month'): ('AMT_REQ_CREDIT_BUREAU_MON',    1),
    ('credit_report',  'bureau_inquiries_quarter'):('AMT_REQ_CREDIT_BUREAU_QRT',   1),
    ('credit_report',  'bureau_inquiries_year'):  ('AMT_REQ_CREDIT_BUREAU_YEAR',   1),
    ('id_document',    'gender'):                 ('CODE_GENDER',                  None),
    ('id_document',    'cnt_children'):           ('CNT_CHILDREN',                 1),
    ('property_doc',   'own_realty'):             ('FLAG_OWN_REALTY',              None),
    ('property_doc',   'own_car'):                ('FLAG_OWN_CAR',                 None),
    ('property_doc',   'housing_type'):           ('NAME_HOUSING_TYPE',            None),
    ('property_doc',   'car_age'):                ('OWN_CAR_AGE',                  1),
    ('property_doc',   'region_rating'):          ('REGION_RATING_CLIENT',         1),
}

ext_by_customer = {}
for rec in flat_results:
    sk_id = rec.get('SK_ID_CURR')
    doc_type = rec.get('doc_type')
    if sk_id is None or doc_type is None:
        continue
    ext_by_customer.setdefault(sk_id, {})
    for field, val in rec.items():
        if field in ('SK_ID_CURR', 'doc_type', 'parse_error'):
            continue
        ext_by_customer[sk_id][(doc_type, field)] = val


def _normalize_categorical(orig_col, ext_val):
    """Normalize LLM-extracted categorical values to match training data labels."""
    val = str(ext_val).strip()
    if orig_col == 'CODE_GENDER':
        v = val.upper()
        if v in ('M', 'MALE'):
            return 'M'
        if v in ('F', 'FEMALE'):
            return 'F'
        return np.nan
    if orig_col in ('FLAG_OWN_REALTY', 'FLAG_OWN_CAR'):
        v = val.upper()
        if v in ('Y', 'YES'):
            return 'Y'
        if v in ('N', 'NO'):
            return 'N'
        return np.nan
    if orig_col == 'NAME_HOUSING_TYPE':
        v = val.lower().replace('/', ' / ')
        v = ' '.join(v.split())
        valid = ['house / apartment', 'rented apartment', 'with parents',
                 'municipal apartment', 'office apartment', 'co-op apartment']
        for label in valid:
            if label in v or v in label:
                return label.title() if label != 'co-op apartment' else 'Co-op apartment'
        if 'house' in v or 'residential' in v or 'condo' in v:
            return 'House / apartment'
        if 'rent' in v:
            return 'Rented apartment'
        if 'apartment' in v or 'apt' in v:
            return 'House / apartment'
        return np.nan
    return ext_val


# Build Model C test set: trimmed features + recovered original columns
reconstructed = trimmed_sample.copy()
recovered_count = 0
skipped_count = 0
for (doc_type, ext_field), (orig_col, scale) in EXTRACTION_TO_ORIGINAL.items():
    if orig_col not in reconstructed.columns:
        reconstructed[orig_col] = np.nan
    for idx in reconstructed.index:
        sk_id = int(reconstructed.at[idx, 'SK_ID_CURR'])
        ext_val = ext_by_customer.get(sk_id, {}).get((doc_type, ext_field))
        if ext_val is not None:
            if scale is not None:
                try:
                    raw = ext_val[0] if isinstance(ext_val, list) else ext_val
                    num_val = float(raw) * scale
                except (ValueError, TypeError):
                    skipped_count += 1
                    continue
                if orig_col == 'AMT_INCOME_TOTAL' and num_val <= 0:
                    skipped_count += 1
                    continue
                reconstructed.at[idx, orig_col] = num_val
            else:
                reconstructed.at[idx, orig_col] = _normalize_categorical(orig_col, ext_val)
            recovered_count += 1

reconstructed.to_csv('../datasets/application_train_reconstructed.csv', index=False)

recovered_cols = sorted(set(reconstructed.columns) - set(trimmed_sample.columns))
print(f"\nReconstructed dataset shape: {reconstructed.shape}")
print(f"  Trimmed columns:   {trimmed_sample.shape[1]}")
print(f"  Recovered columns: {len(recovered_cols)}")
print(f"  Values filled from extraction: {recovered_count}  (skipped: {skipped_count})")
print(f"  Recovered column names: {recovered_cols}")
print(f"Saved to ../datasets/application_train_reconstructed.csv")

Trimmed dataset (full): (307511, 88)
Held-out test set (110 customers): (110, 88)
Training set (rest): (307401, 88)

Reconstructed dataset shape: (110, 104)
  Trimmed columns:   88
  Recovered columns: 16
  Values filled from extraction: 1561  (skipped: 14)
  Recovered column names: ['AMT_REQ_CREDIT_BUREAU_DAY', 'AMT_REQ_CREDIT_BUREAU_HOUR', 'AMT_REQ_CREDIT_BUREAU_MON', 'AMT_REQ_CREDIT_BUREAU_QRT', 'AMT_REQ_CREDIT_BUREAU_WEEK', 'AMT_REQ_CREDIT_BUREAU_YEAR', 'CNT_CHILDREN', 'CODE_GENDER', 'EXT_SOURCE_1', 'EXT_SOURCE_2', 'EXT_SOURCE_3', 'FLAG_OWN_CAR', 'FLAG_OWN_REALTY', 'NAME_HOUSING_TYPE', 'OWN_CAR_AGE', 'REGION_RATING_CLIENT']
Saved to ../datasets/application_train_reconstructed.csv


## Step 4c — Three-Model Comparison

In [7]:
from sklearn.preprocessing import LabelEncoder as _LE

def preprocess_for_lgbm(train_df, test_df):
    """Align columns, encode categoricals, handle DAYS_EMPLOYED anomaly."""
    le = _LE()
    for col in train_df.columns:
        if train_df[col].dtype == 'object':
            if train_df[col].nunique() <= 2:
                combined = pd.concat([train_df[col], test_df[col]], axis=0).astype(str)
                le.fit(combined)
                train_df[col] = le.transform(train_df[col].astype(str))
                test_df[col]  = le.transform(test_df[col].astype(str))
            else:
                train_df = pd.get_dummies(train_df, columns=[col])
                test_df  = pd.get_dummies(test_df, columns=[col])

    train_df, test_df = train_df.align(test_df, join='inner', axis=1)

    if 'DAYS_EMPLOYED' in train_df.columns:
        for df in [train_df, test_df]:
            df['DAYS_EMPLOYED'] = df['DAYS_EMPLOYED'].replace({365243: np.nan})

    return train_df, test_df


def train_and_predict(train_df, train_labels, test_df, model_name="Model"):
    """Train LightGBM on train_df, predict on test_df. Returns (test_preds, feature_importances)."""
    feature_names = list(train_df.columns)
    X_train = np.array(train_df)
    X_test  = np.array(test_df)
    y_train = np.array(train_labels)

    clf = lgb.LGBMClassifier(
        n_estimators=10000, objective='binary',
        class_weight='balanced', learning_rate=0.05,
        reg_alpha=0.1, reg_lambda=0.1,
        subsample=0.8, n_jobs=-1, random_state=50, verbose=-1
    )

    from sklearn.model_selection import train_test_split
    X_tr, X_val, y_tr, y_val = train_test_split(
        X_train, y_train, test_size=0.15, random_state=50, stratify=y_train
    )

    clf.fit(X_tr, y_tr, eval_metric='auc',
            eval_set=[(X_val, y_val)],
            eval_names=['valid'],
            callbacks=[lgb.early_stopping(100), lgb.log_evaluation(200)])

    print(f"  {model_name} best iteration: {clf.best_iteration_}")
    print(f"  {model_name} validation AUC: {clf.best_score_['valid']['auc']:.6f}")

    test_preds = clf.predict_proba(X_test, num_iteration=clf.best_iteration_)[:, 1]
    fi = pd.DataFrame({'feature': feature_names, 'importance': clf.feature_importances_})

    del clf
    gc.collect()
    return test_preds, fi

print("Model training functions defined.")

Model training functions defined.


In [8]:
model_a_auc = float(open('../artifacts/model_a_auc.txt').read().strip())
print(f"Model A benchmark AUC: {model_a_auc:.6f}")

Model A benchmark AUC: 0.765949


In [9]:
# ── Prepare training and test data ──────────────────────────────────────────
# Full structured data for Model A reference prediction on the 56 test IDs
full_train = app_train_full[~app_train_full['SK_ID_CURR'].isin(sample_ids)].copy()
full_test  = app_train_full[app_train_full['SK_ID_CURR'].isin(sample_ids)].copy()

full_train_labels = full_train.pop('TARGET')
full_test_labels  = full_test.pop('TARGET')
full_train.drop(columns=['SK_ID_CURR'], inplace=True)
full_test.drop(columns=['SK_ID_CURR'], inplace=True)

full_train_p, full_test_p = preprocess_for_lgbm(full_train, full_test)

print("=" * 60)
print("Model A — Full structured data")
print("=" * 60)
model_a_preds, model_a_fi = train_and_predict(
    full_train_p, full_train_labels, full_test_p, model_name="Model A"
)
model_a_test_auc = roc_auc_score(full_test_labels, model_a_preds)
print(f"  Model A test AUC on {len(full_test_labels)} held-out customers: {model_a_test_auc:.6f}")

# ── Model B — Trimmed data only ──────────────────────────────────────────────
trim_train_b = trimmed_train.copy()
trim_test_b  = trimmed_sample.copy()

trim_train_b_labels = trim_train_b.pop('TARGET')
trim_test_b_labels  = trim_test_b.pop('TARGET')
trim_train_b.drop(columns=['SK_ID_CURR'], inplace=True)
trim_test_b.drop(columns=['SK_ID_CURR'], inplace=True)

trim_train_b, trim_test_b = preprocess_for_lgbm(trim_train_b, trim_test_b)

print("\n" + "=" * 60)
print("Model B — Trimmed structured data (fields removed)")
print("=" * 60)
model_b_preds, model_b_fi = train_and_predict(
    trim_train_b, trim_train_b_labels, trim_test_b, model_name="Model B"
)
model_b_test_auc = roc_auc_score(trim_test_b_labels, model_b_preds)
print(f"  Model B test AUC on {len(trim_test_b_labels)} held-out customers: {model_b_test_auc:.6f}")

# ── Model C — Full training + recovered features at test time ────────────────
# The model trains on full structured data (same as Model A) so it knows how to
# use features like EXT_SOURCE_1, AMT_INCOME_TOTAL, etc.
# At test time, the 110 customers have their trimmed columns recovered via LLM
# extraction. The gap between Model A and C reflects extraction quality and
# any features that couldn't be recovered from the unstructured documents.

recon_test = reconstructed.copy()
recon_test_labels = recon_test.pop('TARGET')
recon_test.drop(columns=['SK_ID_CURR'], inplace=True)

full_train_c = app_train_full[~app_train_full['SK_ID_CURR'].isin(sample_ids)].copy()
full_train_c_labels = full_train_c.pop('TARGET')
full_train_c.drop(columns=['SK_ID_CURR'], inplace=True)

shared_cols = sorted(set(full_train_c.columns) & set(recon_test.columns))
full_train_c = full_train_c[shared_cols]
recon_test = recon_test[shared_cols]

full_train_c, recon_test = preprocess_for_lgbm(full_train_c, recon_test)

print("\n" + "=" * 60)
print("Model C — Trimmed + LLM-extracted features (recovered columns)")
print("=" * 60)
print(f"  Training features: {full_train_c.shape[1]}")
print(f"  Test features:     {recon_test.shape[1]}")
model_c_preds, model_c_fi = train_and_predict(
    full_train_c, full_train_c_labels, recon_test, model_name="Model C"
)
model_c_test_auc = roc_auc_score(recon_test_labels, model_c_preds)
print(f"  Model C test AUC on {len(recon_test_labels)} held-out customers: {model_c_test_auc:.6f}")

Model A — Full structured data


Training until validation scores don't improve for 100 rounds


[200]	valid's auc: 0.756029	valid's binary_logloss: 0.565232


Early stopping, best iteration is:
[235]	valid's auc: 0.756097	valid's binary_logloss: 0.562082
  Model A best iteration: 235
  Model A validation AUC: 0.756097


  Model A test AUC on 110 held-out customers: 0.823718


...


Model C — Trimmed + LLM-extracted features (recovered columns)
  Training features: 114
  Test features:     114


Training until validation scores don't improve for 100 rounds


[200]	valid's auc: 0.741893	valid's binary_logloss: 0.578184


Early stopping, best iteration is:
[203]	valid's auc: 0.741963	valid's binary_logloss: 0.577838
  Model C best iteration: 203
  Model C validation AUC: 0.741963


  Model C test AUC on 110 held-out customers: 0.833333


## Evaluation Metrics

In [10]:
from scipy.stats import ks_2samp

def compute_ks_statistic(y_true, y_pred):
    """Compute Kolmogorov-Smirnov statistic."""
    pos_scores = y_pred[y_true == 1]
    neg_scores = y_pred[y_true == 0]
    if len(pos_scores) == 0 or len(neg_scores) == 0:
        return 0.0
    stat, _ = ks_2samp(pos_scores, neg_scores)
    return stat

y_true = np.array(full_test_labels)

model_a_ks   = compute_ks_statistic(y_true, model_a_preds)
model_a_gini = 2 * model_a_test_auc - 1

model_b_ks   = compute_ks_statistic(y_true, model_b_preds)
model_b_gini = 2 * model_b_test_auc - 1

model_c_ks   = compute_ks_statistic(y_true, model_c_preds)
model_c_gini = 2 * model_c_test_auc - 1

comparison = pd.DataFrame({
    'Model': ['Model A (Full Structured)', 'Model B (Trimmed Only)', 'Model C (Trimmed + Extracted)'],
    'ROC-AUC': [model_a_test_auc, model_b_test_auc, model_c_test_auc],
    'KS Statistic': [model_a_ks, model_b_ks, model_c_ks],
    'Gini': [model_a_gini, model_b_gini, model_c_gini],
})
comparison['AUC Delta vs A'] = comparison['ROC-AUC'] - model_a_test_auc

os.makedirs('../artifacts', exist_ok=True)
comparison.to_csv('../artifacts/model_comparison.csv', index=False)

print("\n" + "=" * 60)
print("THREE-MODEL COMPARISON")
print("=" * 60)
print(comparison.to_string(index=False))
print(f"\nModel A 5-fold CV AUC (from notebook 01): {model_a_auc:.6f}")
print(f"\nAll results on {len(y_true)} held-out customers.")
print("Saved to ../artifacts/model_comparison.csv")


THREE-MODEL COMPARISON
                        Model  ROC-AUC  KS Statistic     Gini  AUC Delta vs A
    Model A (Full Structured) 0.823718      0.538462 0.647436        0.000000
       Model B (Trimmed Only) 0.766026      0.474359 0.532051       -0.057692
Model C (Trimmed + Extracted) 0.833333      0.573718 0.666667        0.009615

Model A 5-fold CV AUC (from notebook 01): 0.765949

All results on 110 held-out customers.
Saved to ../artifacts/model_comparison.csv


## Extraction Accuracy vs Ground Truth

In [11]:
# Build a lookup keyed by (SK_ID_CURR, doc_type) → field values (preserving strings)
ext_lookup = {}
for rec in flat_results:
    key = (rec.get('SK_ID_CURR'), rec.get('doc_type'))
    ext_lookup[key] = rec

ground_truth = app_train_full[app_train_full['SK_ID_CURR'].isin(sample_ids)].copy()

# Mapping: (doc_type, extracted_field) → (ground truth column, type, scale_factor)
# scale_factor: multiply extracted value by this before comparing to ground truth
# e.g. bank statement has monthly_income but ground truth AMT_INCOME_TOTAL is annual
FIELD_MAP = {
    ('id_document',    'gender'):          ('CODE_GENDER',         'cat', 1),
    ('bank_statement', 'monthly_income'):  ('AMT_INCOME_TOTAL',   'num', 12),
    ('credit_report',  'ext_source_1'):    ('EXT_SOURCE_1',       'num', 1),
    ('credit_report',  'ext_source_2'):    ('EXT_SOURCE_2',       'num', 1),
    ('credit_report',  'ext_source_3'):    ('EXT_SOURCE_3',       'num', 1),
    ('property_doc',   'own_realty'):       ('FLAG_OWN_REALTY',    'cat', 1),
    ('property_doc',   'own_car'):          ('FLAG_OWN_CAR',       'cat', 1),
    ('property_doc',   'housing_type'):     ('NAME_HOUSING_TYPE',  'cat', 1),
}

cat_correct, cat_total = 0, 0
num_correct, num_total = 0, 0
accuracy_rows = []

for _, gt_row in ground_truth.iterrows():
    sk_id = int(gt_row['SK_ID_CURR'])

    for (doc_type, ext_field), (gt_col, ftype, scale) in FIELD_MAP.items():
        rec = ext_lookup.get((sk_id, doc_type))
        if rec is None or ext_field not in rec:
            continue
        ext_val = rec[ext_field]
        gt_val  = gt_row.get(gt_col)
        if ext_val is None or pd.isna(gt_val):
            continue

        if ftype == 'cat':
            cat_total += 1
            match = str(ext_val).strip().upper() == str(gt_val).strip().upper()
            if match:
                cat_correct += 1
            accuracy_rows.append({
                'SK_ID_CURR': sk_id, 'field': f'{ext_field}_{doc_type}',
                'extracted': ext_val, 'ground_truth': gt_val,
                'type': 'categorical', 'match': match
            })
        else:
            num_total += 1
            try:
                raw = ext_val[0] if isinstance(ext_val, list) else ext_val
                ext_num = float(raw) * scale
                gt_num  = float(gt_val)
                within_tol = abs(ext_num - gt_num) <= 0.10 * abs(gt_num) if gt_num != 0 else ext_num == 0
            except (ValueError, TypeError):
                ext_num = None
                within_tol = False
            if within_tol:
                num_correct += 1
            accuracy_rows.append({
                'SK_ID_CURR': sk_id, 'field': f'{ext_field}_{doc_type}',
                'extracted': ext_val, 'ground_truth': gt_val,
                'scaled_extracted': ext_num,
                'type': 'numeric', 'match': within_tol
            })

cat_acc = cat_correct / cat_total * 100 if cat_total > 0 else 0
num_acc = num_correct / num_total * 100 if num_total > 0 else 0

print("=" * 60)
print("EXTRACTION ACCURACY vs GROUND TRUTH")
print("=" * 60)
print(f"  Categorical fields: {cat_correct}/{cat_total} correct ({cat_acc:.1f}%)")
print(f"    Target: >= 85%")
print(f"  Numeric fields:     {num_correct}/{num_total} within ±10% ({num_acc:.1f}%)")
print(f"    Target: >= 80%")

accuracy_df = pd.DataFrame(accuracy_rows)
accuracy_df.to_csv('../artifacts/extraction_accuracy.csv', index=False)
print(f"\nDetailed accuracy saved to ../artifacts/extraction_accuracy.csv")

EXTRACTION ACCURACY vs GROUND TRUTH
  Categorical fields: 364/398 correct (91.5%)
    Target: >= 85%
  Numeric fields:     286/305 within ±10% (93.8%)
    Target: >= 80%

Detailed accuracy saved to ../artifacts/extraction_accuracy.csv


## Success Criteria Validation

In [12]:
print("=" * 60)
print("SUCCESS CRITERIA CHECK")
print("=" * 60)

auc_delta = abs(model_a_test_auc - model_c_test_auc)

criteria = {
    "Document generation success rate >= 95%": None,
    "LLM extraction valid JSON parse rate >= 90%": (
        f"{parse_success}/{total} ({parse_success/total*100:.1f}%)" if total > 0 else "N/A"
    ),
    "Extraction accuracy — categorical >= 85%": f"{cat_correct}/{cat_total} ({cat_acc:.1f}%)",
    "Extraction accuracy — numeric >= 80% (±10%)": f"{num_correct}/{num_total} ({num_acc:.1f}%)",
    "ROC-AUC delta (A vs C) < 0.02": f"{auc_delta:.6f} ({'PASS' if auc_delta < 0.02 else 'FAIL'})",
    "Model B ROC-AUC > 0.70": f"{model_b_test_auc:.6f} ({'PASS' if model_b_test_auc > 0.70 else 'FAIL — small test set'})",
}

gen_log_path = '../unstructured_data/generation_log.jsonl'
if os.path.exists(gen_log_path):
    gen_logs = [json.loads(line) for line in open(gen_log_path)]
    gen_success = sum(1 for l in gen_logs if l.get('success'))
    gen_total_docs = len(gen_logs)
    gen_rate = gen_success / gen_total_docs * 100 if gen_total_docs > 0 else 0
    criteria["Document generation success rate >= 95%"] = f"{gen_success}/{gen_total_docs} ({gen_rate:.1f}%)"

for criterion, result in criteria.items():
    status = result if result else "Not yet evaluated"
    passed = 'PASS' in str(status) or (
        '%' in str(status) and float(str(status).split('(')[1].split('%')[0]) >= 80
    ) if status != "Not yet evaluated" else False
    mark = "PASS" if passed else "REVIEW"
    print(f"  [{mark}] {criterion}")
    print(f"         {status}")
    print()

print(f"Tested on {len(sample_ids)} held-out customers.")

SUCCESS CRITERIA CHECK
  [PASS] Document generation success rate >= 95%
         715/716 (99.9%)

  [PASS] LLM extraction valid JSON parse rate >= 90%
         656/660 (99.4%)

  [PASS] Extraction accuracy — categorical >= 85%
         364/398 (91.5%)

  [PASS] Extraction accuracy — numeric >= 80% (±10%)
         286/305 (93.8%)

  [PASS] ROC-AUC delta (A vs C) < 0.02
         0.009615 (PASS)

  [PASS] Model B ROC-AUC > 0.70
         0.766026 (PASS)

Tested on 110 held-out customers.


## Update Documentation

In [13]:
from docx import Document
from docx.shared import Pt, Inches, RGBColor
from docx.enum.text import WD_PARAGRAPH_ALIGNMENT
from datetime import datetime

DOC_PATH = '../docs/SyntheticDataForBNPL.docx'
doc = Document(DOC_PATH)

# ── Remove previous "PoC Results" section if it exists (idempotent) ────────
result_heading_idx = None
for i, p in enumerate(doc.paragraphs):
    if p.text.strip() == 'PoC Results' and p.style.name.startswith('Heading'):
        result_heading_idx = i
        break

if result_heading_idx is not None:
    # Delete from that heading to end
    while len(doc.paragraphs) > result_heading_idx:
        p = doc.paragraphs[result_heading_idx]
        p._element.getparent().remove(p._element)

# ── Append new "PoC Results" section ───────────────────────────────────────
doc.add_heading('PoC Results', level=1)
doc.add_paragraph(f'Generated on {datetime.now().strftime("%Y-%m-%d %H:%M")} | '
                  f'Sample size: {len(sample_ids)} customers | '
                  f'6 document types per customer')

# Model Comparison Table
doc.add_heading('Model Comparison', level=2)
tbl = doc.add_table(rows=1, cols=5, style='TableNormal')
hdr = tbl.rows[0].cells
for j, h in enumerate(['Model', 'ROC-AUC', 'KS Statistic', 'Gini', 'AUC Delta vs A']):
    hdr[j].text = h

models_data = [
    ('Model A (Full Structured)', model_a_test_auc, model_a_ks, model_a_gini, 0.0),
    ('Model B (Trimmed Only)',    model_b_test_auc, model_b_ks, model_b_gini, model_b_test_auc - model_a_test_auc),
    ('Model C (Trimmed + Extracted)', model_c_test_auc, model_c_ks, model_c_gini, model_c_test_auc - model_a_test_auc),
]

for name, auc, ks, gini, delta in models_data:
    row = tbl.add_row().cells
    row[0].text = name
    row[1].text = f'{auc:.6f}'
    row[2].text = f'{ks:.6f}'
    row[3].text = f'{gini:.6f}'
    row[4].text = f'{delta:+.6f}'

doc.add_paragraph(f'Model A 5-fold CV AUC (from notebook 01): {model_a_auc:.6f}')

# Extraction Accuracy
doc.add_heading('Extraction Accuracy vs Ground Truth', level=2)
doc.add_paragraph(f'Categorical accuracy: {cat_correct}/{cat_total} ({cat_acc:.1f}%) — target >= 85%')
doc.add_paragraph(f'Numeric accuracy (±10%): {num_correct}/{num_total} ({num_acc:.1f}%) — target >= 80%')

# Generation Stats
doc.add_heading('Document Generation', level=2)
if os.path.exists(gen_log_path):
    doc.add_paragraph(f'Success rate: {gen_success}/{gen_total_docs} ({gen_rate:.1f}%) — target >= 95%')
else:
    doc.add_paragraph('Generation log not found.')

extraction_rate = parse_success / total * 100 if total > 0 else 0
doc.add_paragraph(f'LLM extraction parse rate: {parse_success}/{total} ({extraction_rate:.1f}%) — target >= 90%')

# Success Criteria Summary
doc.add_heading('Success Criteria Summary', level=2)
criteria_tbl = doc.add_table(rows=1, cols=3, style='TableNormal')
ch = criteria_tbl.rows[0].cells
ch[0].text = 'Criterion'
ch[1].text = 'Result'
ch[2].text = 'Status'

for criterion, result in criteria.items():
    status_str = result if result else "Not evaluated"
    passed = 'PASS' in str(status_str)
    row = criteria_tbl.add_row().cells
    row[0].text = criterion
    row[1].text = str(status_str)
    row[2].text = 'PASS' if passed else 'REVIEW'

doc.save(DOC_PATH)
print(f"Documentation updated: {DOC_PATH}")

Documentation updated: ../docs/SyntheticDataForBNPL.docx
